In [24]:
!pip install openai pypdf -q

In [25]:
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("❌ API key not found in Colab Secrets")

OPENROUTER_API_KEY = OPENROUTER_API_KEY.strip()  # IMPORTANT FIX
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

In [26]:
from pypdf import PdfReader

def read_pdf(file):
    reader = PdfReader(file)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

In [27]:
def optimizer_agent(resume_text, jd_text):
    prompt = f"""
You are the Optimizer Agent for TalentCheck AI.

Return ONLY valid JSON:

{{
  "match_score": 0-100,
  "key_skills": ["skill1", "skill2"],
  "summary": "short explanation"
}}

RESUME:
{resume_text}

JOB DESCRIPTION:
{jd_text}
"""

    res = client.chat.completions.create(
        model="openai/gpt-oss-120b:free",
        messages=[{"role": "user", "content": prompt}]
    )

    return res.choices[0].message.content

In [28]:
def evaluator_agent(scorecard, resume_text):
    prompt = f"""
You are a strict technical auditor.

TASK:
Check for:
1. timeline errors
2. hallucinated skills
3. seniority mismatch

RULES:
- Be strict but factual
- Do NOT over-explain
- Do NOT add extra assumptions
- Ignore phone numbers or irrelevant text
- Focus only on resume content vs scorecard

OUTPUT FORMAT (VERY IMPORTANT):
If no issues:
PASS

If issues exist:
FAIL
- bullet 1
- bullet 2
- bullet 3

RESUME:
{resume_text}

SCORECARD:
{scorecard}
"""

    res = client.chat.completions.create(
        model="google/gemma-3-4b-it:free",
        messages=[{"role": "user", "content": prompt}]
    )

    return res.choices[0].message.content

In [29]:
def refine(resume_text, jd_text, max_iter=3):

    scorecard = optimizer_agent(resume_text, jd_text)

    for i in range(max_iter):

        result = evaluator_agent(scorecard, resume_text)

        if "PASS" in result:
            return {
                "verified_scorecard": scorecard,
                "status": "PASSED",
                "audit": result
            }

        # feedback loop
        scorecard = optimizer_agent(
            resume_text + "\nFEEDBACK:\n" + result,
            jd_text
        )

    return {
        "verified_scorecard": scorecard,
        "status": "FAILED_AFTER_RETRIES",
        "audit": result
    }

In [30]:
from google.colab import files

# =========================
# UPLOAD CV
# =========================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

resume_text = read_pdf(file_name)

# =========================
# USER INPUT JOB DESCRIPTION
# =========================
print("\n✍️ Paste Job Description below:\n")
jd_text = input("JD: ")

# =========================
# RUN PIPELINE
# =========================
result = refine(resume_text, jd_text)

print("\n================ RESULT ================\n")
print(result)

Saving Salma_Kasssem (3).pdf to Salma_Kasssem (3).pdf

✍️ Paste Job Description below:

JD: We are hiring a Python AI Engineer to join our growing AI team.  Responsibilities: - Design and develop machine learning models for real-world applications - Build and deploy AI/ML pipelines into production environments - Work with large datasets for data preprocessing, cleaning, and feature engineering - Collaborate with data scientists and software engineers to improve model performance - Optimize models for scalability and performance  Requirements: - Strong Python programming skills - Experience with machine learning frameworks such as Scikit-learn, TensorFlow, or PyTorch - Knowledge of data processing libraries like Pandas and NumPy - Understanding of model deployment (Docker, APIs, cloud platforms like AWS or GCP) - Experience working on AI/ML projects or internships  Nice to have: - Experience with deep learning models - Familiarity with MLOps practices - Knowledge of cloud-based AI deplo